In [3]:
from roboflow import Roboflow
rf = Roboflow(api_key="KebUB2aZYK0JETrujiWA")
project = rf.workspace("vehicledetection-ioag0").project("vehicle-detection-mk4lt")
version = project.version(3)
dataset = version.download("yolov8", location="datasets/v2_full_res_unlabelled")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to datasets/v2_full_res_unlabelled in yolov8:: 100%|██████████| 22542/22542 [00:09<00:00, 2478.87it/s]


In [4]:
from roboflow import Roboflow
rf = Roboflow(api_key="KebUB2aZYK0JETrujiWA")
project = rf.workspace("vehicledetection-ioag0").project("vehicle-detection-model-labelled-lower-iou")
version = project.version(1)
dataset = version.download("yolov8", location="datasets/v2_640px_labelled")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to datasets/v2_640px_labelled in yolov8:: 100%|██████████| 22015/22015 [00:08<00:00, 2678.16it/s]


In [4]:
import os
import shutil
import yaml

# --- Configuration ---
base_dir = "datasets"

labelled_640_dir = os.path.join(base_dir, "v2_640px_labelled")
unlabelled_full_res_dir = os.path.join(base_dir, "v2_full_res_unlabelled")
output_full_res_labelled_dir = os.path.join(base_dir, "v2_full_res_labelled")

# Source directories
source_labelled_images_dir = os.path.join(labelled_640_dir, "train", "images")
source_labelled_labels_dir = os.path.join(labelled_640_dir, "train", "labels")
source_unlabelled_images_dir = os.path.join(unlabelled_full_res_dir, "train", "images")
source_yaml_file = os.path.join(labelled_640_dir, "data.yaml")

# Destination directories
dest_images_dir = os.path.join(output_full_res_labelled_dir, "train", "images")
dest_labels_dir = os.path.join(output_full_res_labelled_dir, "train", "labels")
dest_yaml_file = os.path.join(output_full_res_labelled_dir, "data.yaml")

# --- Helper Functions ---
def ensure_dir_exists(directory_path):
    """Creates a directory if it doesn't exist."""
    os.makedirs(directory_path, exist_ok=True)

def get_base_key_from_filename(filename):
    """
    Extracts the base key from a filename like '12957_png.rf.hash.jpg'.
    Returns the part before '.rf.', e.g., '12957_png'.
    If '.rf.' is not found, it returns the filename without extension as a fallback,
    though the problem implies '.rf.' will be present.
    """
    parts = filename.split('.rf.', 1) # Split on the first occurrence of .rf.
    if len(parts) > 0:
        return parts[0]
    # Fallback, though not expected given the filename format
    return os.path.splitext(filename)[0]


# --- Main Script ---
def process_datasets():
    print("Starting dataset processing with '.rf.' based matching...")

    # 1. Create destination directories
    ensure_dir_exists(dest_images_dir)
    ensure_dir_exists(dest_labels_dir)
    print(f"Ensured output directories exist:\n  {dest_images_dir}\n  {dest_labels_dir}")

    # --- Counters for summary ---
    copied_files_count = 0
    skipped_missing_source_label_count = 0  # Labels missing for 640px images
    skipped_unlabelled_not_in_labelled_map = 0 # Full-res images with no matching base_key in labelled set
    skipped_labelled_key_already_exists = 0 # If multiple 640px images map to the same base_key

    # 2. Build a map of labelled data from the 640px dataset
    #    Map structure: {base_key: {'original_image_filename': str, 'source_label_filename': str}}
    labelled_data_map = {}
    print(f"\nScanning 640px labelled images in: {source_labelled_images_dir}")
    try:
        labelled_image_files = [f for f in os.listdir(source_labelled_images_dir)
                                if os.path.isfile(os.path.join(source_labelled_images_dir, f))]
    except FileNotFoundError:
        print(f"ERROR: Source directory not found: {source_labelled_images_dir}")
        return

    for img_filename_640 in labelled_image_files:
        base_key = get_base_key_from_filename(img_filename_640)
        
        # Derive the label filename from the original 640px image filename
        label_filename_640 = os.path.splitext(img_filename_640)[0] + ".txt"
        source_label_path_640 = os.path.join(source_labelled_labels_dir, label_filename_640)

        if not os.path.exists(source_label_path_640):
            print(f"  WARNING: Label file '{label_filename_640}' for 640px image '{img_filename_640}' not found. Skipping this entry for map.")
            skipped_missing_source_label_count += 1
            continue

        if base_key in labelled_data_map:
            old_entry = labelled_data_map[base_key]['original_image_filename']
            print(f"  WARNING: Duplicate base key '{base_key}' from '{img_filename_640}'. It was already mapped by '{old_entry}'. Overwriting with latest.")
            skipped_labelled_key_already_exists += 1
        
        labelled_data_map[base_key] = {
            'original_image_filename': img_filename_640, # Full original filename, e.g., key.rf.xxxx.jpg
            'source_label_filename': label_filename_640    # Full original label filename, e.g., key.rf.xxxx.txt
        }
    print(f"Built map for {len(labelled_data_map)} unique base keys from {len(labelled_image_files)} 640px labelled images.")
    if skipped_missing_source_label_count > 0:
        print(f"  ({skipped_missing_source_label_count} images were skipped due to missing label files in 640px set.)")
    if skipped_labelled_key_already_exists > 0:
        print(f"  ({skipped_labelled_key_already_exists} duplicate base keys were encountered and overwritten in 640px set.)")


    # 3. Iterate through unlabelled full-resolution images and process if base_key matches
    print(f"\nScanning full-resolution unlabelled images in: {source_unlabelled_images_dir}")
    try:
        unlabelled_full_res_image_files = [f for f in os.listdir(source_unlabelled_images_dir)
                                           if os.path.isfile(os.path.join(source_unlabelled_images_dir, f))]
    except FileNotFoundError:
        print(f"ERROR: Source directory not found: {source_unlabelled_images_dir}")
        if not labelled_data_map: print("No labelled data was mapped either.")
        return
    
    if not unlabelled_full_res_image_files:
        print(f"No images found in {source_unlabelled_images_dir}. Processing will stop.")


    for full_res_img_filename in unlabelled_full_res_image_files: # e.g., 12957_png.rf.yyyy.jpg
        current_base_key = get_base_key_from_filename(full_res_img_filename)

        if current_base_key in labelled_data_map:
            labelled_info = labelled_data_map[current_base_key]
            
            # Path to the original label file from the 640px dataset
            source_label_to_copy_path = os.path.join(source_labelled_labels_dir, labelled_info['source_label_filename'])
            
            # Path to the full-resolution image to copy
            source_full_res_image_to_copy_path = os.path.join(source_unlabelled_images_dir, full_res_img_filename)

            # Destination paths: use the full-res image's original filename for both image and derived label name
            dest_image_path = os.path.join(dest_images_dir, full_res_img_filename)
            dest_label_filename = os.path.splitext(full_res_img_filename)[0] + ".txt"
            dest_label_path = os.path.join(dest_labels_dir, dest_label_filename)

            # Double-check that the source label file still exists (should be true from map building)
            if not os.path.exists(source_label_to_copy_path):
                print(f"  ERROR (UNEXPECTED): Source label file '{source_label_to_copy_path}' for base key '{current_base_key}' "
                      f"(mapped from 640px image '{labelled_info['original_image_filename']}') "
                      f"not found during copy phase. Skipping '{full_res_img_filename}'.")
                continue

            # Copy full-resolution image
            try:
                shutil.copy2(source_full_res_image_to_copy_path, dest_image_path)
            except Exception as e:
                print(f"  ERROR: Could not copy image '{source_full_res_image_to_copy_path}' to '{dest_image_path}': {e}")
                continue # Skip this file

            # Copy the corresponding label file (content from 640px label, name derived from full-res image)
            try:
                shutil.copy2(source_label_to_copy_path, dest_label_path)
                copied_files_count += 1
                print(f"  Processed: Base key '{current_base_key}'.")
                print(f"    Copied full-res image: '{full_res_img_filename}'")
                print(f"    Copied label from:     '{labelled_info['source_label_filename']}' as '{dest_label_filename}'")
            except Exception as e:
                print(f"  ERROR: Could not copy label '{source_label_to_copy_path}' to '{dest_label_path}': {e}")
                # Clean up copied image if label copy fails to maintain consistency
                if os.path.exists(dest_image_path):
                    try:
                        os.remove(dest_image_path)
                        print(f"    Cleaned up partially copied image: '{dest_image_path}'")
                    except OSError as oe:
                        print(f"    ERROR: Could not clean up image '{dest_image_path}': {oe}")
        else:
            # This full-res image's base key is not in the labelled_data_map
            # This means it's in unlabelled but not in labelled, so ignore it.
            print(f"  INFO: Full-res image '{full_res_img_filename}' (base key '{current_base_key}') has no matching entry in the 640px labelled set map. Skipping.")
            skipped_unlabelled_not_in_labelled_map += 1


    print(f"\n--- Processing Summary ---")
    print(f"Successfully processed and copied {copied_files_count} image-label pairs to '{output_full_res_labelled_dir}'.")
    if skipped_missing_source_label_count > 0:
        print(f"Skipped {skipped_missing_source_label_count} entries during 640px map building due to missing original label files.")
    if skipped_labelled_key_already_exists > 0:
         print(f"Encountered and overwrote {skipped_labelled_key_already_exists} duplicate base keys during 640px map building (last one was kept).")
    if skipped_unlabelled_not_in_labelled_map > 0:
        print(f"Skipped {skipped_unlabelled_not_in_labelled_map} full-resolution images as their base keys were not found in the 640px labelled set map.")


    # 4. Handle data.yaml
    if not os.path.exists(source_yaml_file):
        print(f"\nERROR: Source 'data.yaml' not found at {source_yaml_file}. Cannot create new YAML for the output dataset.")
    else:
        try:
            with open(source_yaml_file, 'r') as f:
                data_yaml = yaml.safe_load(f)

            # Update paths. If data.yaml is in output_full_res_labelled_dir:
            # 'path' can be relative to project or absolute.
            # 'train', 'val', 'test' paths should be relative to the data.yaml file itself.
            data_yaml['path'] = os.path.abspath(output_full_res_labelled_dir) # Provides an absolute path to the dataset root
            data_yaml['train'] = 'train/images' # Relative to the new data.yaml location
            
            # Remove 'val' or 'test' keys if they are not relevant for the new dataset or set to new paths.
            # For this script, we are only creating a 'train' set.
            if 'val' in data_yaml:
                data_yaml.pop('val', None)
                print(f"\nRemoved 'val' key from the new data.yaml.")
            if 'test' in data_yaml:
                data_yaml.pop('test', None)
                print(f"Removed 'test' key from the new data.yaml.")

            # Roboflow specific keys can be kept or removed.
            # If keeping, you might want to update version details if applicable.
            # if 'roboflow' in data_yaml:
            #     data_yaml['roboflow']['version'] = # new version if any
            #     pass

            with open(dest_yaml_file, 'w') as f:
                yaml.dump(data_yaml, f, sort_keys=False, default_flow_style=None)
            print(f"\nUpdated and saved 'data.yaml' to: {dest_yaml_file}")
            print(f"  New 'path' in YAML: {data_yaml.get('path')}")
            print(f"  New 'train' path in YAML: {data_yaml.get('train')}")

        except Exception as e:
            print(f"\nERROR: Could not process 'data.yaml': {e}")

    print("\nDataset processing finished.")

if __name__ == "__main__":
    process_datasets()

Starting dataset processing with '.rf.' based matching...
Ensured output directories exist:
  datasets\v2_full_res_labelled\train\images
  datasets\v2_full_res_labelled\train\labels

Scanning 640px labelled images in: datasets\v2_640px_labelled\train\images
Built map for 11005 unique base keys from 11005 640px labelled images.

Scanning full-resolution unlabelled images in: datasets\v2_full_res_unlabelled\train\images


  Processed: Base key '10000_png'.
    Copied full-res image: '10000_png.rf.700af20ad81758ad11a2364b80fecd67.jpg'
    Copied label from:     '10000_png.rf.7388d1ec9ee5da525ab3bfea55654d7a.txt' as '10000_png.rf.700af20ad81758ad11a2364b80fecd67.txt'
  Processed: Base key '10001_png'.
    Copied full-res image: '10001_png.rf.77dbaa400c0f428d2e3c998f3811e186.jpg'
    Copied label from:     '10001_png.rf.72c8bde3597ada2b764681a14bc32a32.txt' as '10001_png.rf.77dbaa400c0f428d2e3c998f3811e186.txt'
  Processed: Base key '10002_png'.
    Copied full-res image: '10002_png.rf.e06944b2f2ea81166daafa1190eaa4a0.jpg'
    Copied label from:     '10002_png.rf.69ec6a47bb9ecd00a72b0763db371df0.txt' as '10002_png.rf.e06944b2f2ea81166daafa1190eaa4a0.txt'
  Processed: Base key '10003_png'.
    Copied full-res image: '10003_png.rf.723afe73d74b9982f6eefbc47d493dc3.jpg'
    Copied label from:     '10003_png.rf.112d04aadd26f559d80241a1a54f5bca.txt' as '10003_png.rf.723afe73d74b9982f6eefbc47d493dc3.txt'
  Proces

In [6]:
len(os.listdir(r'C:\Users\swest\Desktop\traffic-capturing\datasets\v2_full_res_labelled\train\labels'))

11005